# BCPC surface: a convex B-spline surface in PLS space

For each model this notebook reads the full-fit BCPC bundle that `notebooks/2_bcpc_out_of_fold.ipynb` saved in `artifacts/<model>/bcpc/` and takes its `bcpc_arc_length` from it. It does not read the stakes classes.

1. **PLS.** Centred, unscaled PLS components against `bcpc_arc_length`, not rotated.
2. **Surface.** A smooth convex surface in all the components, fitted by orthogonal-distance least squares: each row counts by its squared distance to its closest point on the surface. The surface is a graph of cubic tensor B-spline heights over the rows' weighted principal plane. Its height along one normal direction, the rows' mean-curvature direction, is held convex, so the surface lies on the boundary of its own convex hull. The other heights may bend either way, so a saddle is allowed. A thin-plate bending penalty keeps it smooth. The arc length is not used in the fit.

The fit alternates between the rows' closest points (warm-started Newton steps) and the heights. The heights share one design matrix, so each step is one Cholesky solve plus a small constrained solve for the convex height. The surface is drawn trimmed to the convex hull of the rows' closest points.

Every template file gets equal total weight, split equally among its rows. Nothing is saved; the fitted surface is drawn in PLS1-3.

In [2]:
from pathlib import Path
from dataclasses import replace
import gc
import sys
import time
import traceback

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'bcpc_surface.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import bcpc_surface as bs
from scripts import geometric_surface as gs
from scripts.pipeline_config import discover_run_dirs

print('Repository:', ROOT)

Repository: c:\Users\91967\Desktop\AISC\stakes_manifold


## Configuration

`MODELS = None` fits every model directory under `ARTIFACT_ROOT` that has a BCPC bundle (`bcpc/rows.parquet`); a list of directory names restricts the run to those models.

- `PLS_COMPONENTS`: how many PLS components the surface is fitted in. The section *Choose the number of PLS components* below recommends a value per model.
- `KNOTS`: interior knots of the B-spline heights along the chart's two axes (the principal plane's first and second axes). More knots follow finer structure.
- `REGULARIZATION`: weight of the thin-plate bending penalty. The row weights sum to one and the energy is taken in plane units times the box area, so the loss is the weighted mean squared distance plus `REGULARIZATION` times the bending energy, both in squared PLS units. Larger values give a flatter surface.
- `OUTER_WEIGHT`: slightly raises the weight of rows far from the cloud's centroid in the surface fit. Each row's weight is multiplied by 1 + `OUTER_WEIGHT` × r / r_rms, where r is its distance from the weighted centroid in PLS space and r_rms the weighted RMS of those distances. So with 0.2 a row at the typical distance counts 1.2 times as much as one at the centroid, and one twice as far out 1.4 times. 0 leaves the template weighting unchanged. PLS is not affected. The distance and variance diagnostics are reported in the unboosted weights, so runs with different values compare directly.
- `PER_MODEL`: per-model values of `PLS_COMPONENTS`, `REGULARIZATION` and `OUTER_WEIGHT`, keyed by model directory name. A model that is missing, or a key left out, takes the default above. An unknown model name or key raises before anything is fitted.
- `TOLERANCE`: the fit stops when an iteration lowers the loss by less than this share of it.

`INCLUDE_EXCLUDED_TEMPLATES` adds the rows of `cv.EXCLUDED_TEMPLATES` to PLS and the surface, with the arc lengths saved for them in `bcpc/excluded_rows.parquet`. `PLOT_ROWS` only thins the plot.

Runtime per model is about 25-35 s to load the activations and fit PLS, then about 5 s for the surface. The component selection takes about 1.5 minutes per model.

In [3]:
ARTIFACT_ROOT = ROOT / 'artifacts' / 'content' / 'artifacts'
MODELS = None                       # None = every model with a BCPC bundle
INCLUDE_EXCLUDED_TEMPLATES = False

PLS_COMPONENTS = 16                  # default number of PLS components
KNOTS = (6, 4)                      # interior knots along the chart's two axes
REGULARIZATION = 5e-5               # default bending-penalty weight
OUTER_WEIGHT = 1.0                  # default extra weight per RMS distance from the centroid (0 = none)
TOLERANCE = 1e-6                    # relative loss improvement at which the fit stops

# Per-model overrides of PLS_COMPONENTS, REGULARIZATION and OUTER_WEIGHT; anything left out takes the default.
PER_MODEL = {}
# PER_MODEL = {
#     'gemma-4-31B-it': dict(PLS_COMPONENTS=16, REGULARIZATION=5e-5, OUTER_WEIGHT=1.0),
#     'Mistral-Small-3.1-24B-Instruct-2503': dict(REGULARIZATION=5e-5, OUTER_WEIGHT=1.0),
#     'Qwen3-32B': dict(REGULARIZATION=1e-3, OUTER_WEIGHT=0.2),
# }

SURFACE = gs.SurfaceConfig(shape_model='convex', shape_knots=KNOTS,
                           shape_bending=REGULARIZATION, outer_weight=OUTER_WEIGHT,
                           fit_tolerance=TOLERANCE)
SETTINGS = bs.SurfaceSettings(pls_components=PLS_COMPONENTS, surface=SURFACE,
                              include_excluded_templates=INCLUDE_EXCLUDED_TEMPLATES)
PLOT_ROWS = 8_000                   # rows drawn in the plot; None draws all

# Where each override goes: the surface config, or the settings themselves.
OVERRIDES = {'REGULARIZATION': 'shape_bending', 'OUTER_WEIGHT': 'outer_weight'}
SETTING_OVERRIDES = {'PLS_COMPONENTS': 'pls_components'}
allowed = {*OVERRIDES, *SETTING_OVERRIDES}
unknown = {name: sorted(set(values) - allowed) for name, values in PER_MODEL.items()
           if set(values) - allowed}
if unknown:
    raise ValueError(f'PER_MODEL takes only {sorted(allowed)}; unknown keys: {unknown}')


def settings_for(model):
    """SETTINGS with the model's PER_MODEL overrides applied."""
    values = PER_MODEL.get(model, {})
    surface = replace(SURFACE, **{OVERRIDES[k]: v for k, v in values.items() if k in OVERRIDES})
    return replace(SETTINGS, surface=surface,
                   **{SETTING_OVERRIDES[k]: v for k, v in values.items() if k in SETTING_OVERRIDES})

## Models

Every model under `ARTIFACT_ROOT` with a BCPC bundle, or the ones `MODELS` names.

In [4]:
run_dirs = discover_run_dirs(ARTIFACT_ROOT)
if MODELS is not None:
    missing = [name for name in MODELS
               if ARTIFACT_ROOT / name not in run_dirs or not bs.has_bcpc_bundle(ARTIFACT_ROOT / name)]
    if missing:
        raise ValueError(f'No cached run with a BCPC bundle for {missing} under {ARTIFACT_ROOT}.')
    run_dirs = [ARTIFACT_ROOT / name for name in MODELS]
skipped = [path.name for path in run_dirs if not bs.has_bcpc_bundle(path)]
run_dirs = [path for path in run_dirs if bs.has_bcpc_bundle(path)]
if skipped:
    print('No BCPC bundle, skipped:', ', '.join(skipped))
stray = sorted(set(PER_MODEL) - {path.name for path in discover_run_dirs(ARTIFACT_ROOT)})
if stray:
    raise ValueError(f'PER_MODEL names models with no run under {ARTIFACT_ROOT}: {stray}')
if not run_dirs:
    raise ValueError(f'No model under {ARTIFACT_ROOT} has a BCPC bundle; '
                     'run notebooks/2_bcpc_out_of_fold.ipynb first.')
print(f'{len(run_dirs)} model(s):', ', '.join(path.name for path in run_dirs))

3 model(s): gemma-4-31B-it, Mistral-Small-3.1-24B-Instruct-2503, Qwen3-32B


## Choose the number of PLS components

The surface is fitted in the first k PLS components, so k should keep the directions that carry arc-length information beyond the rows they were fitted on, and drop the ones that only fit noise. This cell measures that by cross-validation, then applies a standard, pre-stated rule.

- **Criterion.** How well the first k components predict `bcpc_arc_length`, the BCPC's continuous output, on held-out tasks. PLS is refitted on the other tasks exactly as the surface's PLS is fitted, with the same equal-template weights, and the held-out error is the weighted mean squared error. The stakes classes are not used.
- **Folds.** Each fold holds out whole tasks, every template of a task together, so a held-out task is never seen in any wording. Tasks are dealt to folds stratified by their mean `bcpc_arc_length`, so every fold spans the whole range. 5 folds, repeated 3 times with different draws.
- **Rule.** The one-standard-error rule (Hastie, Tibshirani and Friedman, *The Elements of Statistical Learning*, section 7.10): the fewest components whose mean held-out error is within one standard error of the lowest. It prefers the smaller space when the extra components' gain is within the noise of the estimate.

The table also shows `gain`, how much the k-th component lowers held-out error on the same folds, with its paired standard error `gain_se`. A component whose `gain` is a few `gain_se` above zero still helps on unseen tasks, even if its gain is too small to clear the one-SE rule. Also shown: `q2` (held-out R2 against the training mean), the in-sample R2, and the share of feature variance the first k components hold.

Each training fit comes from the full data's weighted cross-products minus the held-out fold's, so the activations are read once per model. The recommendations are printed at the end. Set them in `PER_MODEL` (or `PLS_COMPONENTS`) and rerun the configuration cell before fitting.

In [ ]:
SELECTION_MAX_COMPONENTS = 30       # largest component count tried
SELECTION_FOLDS = 5
SELECTION_REPEATS = 3

selections = {}
for run_dir in run_dirs:
    started = time.perf_counter()
    summary, fold_table = bs.select_pls_components(
        run_dir, settings_for(run_dir.name), max_components=SELECTION_MAX_COMPONENTS,
        folds=SELECTION_FOLDS, repeats=SELECTION_REPEATS)
    selections[run_dir.name] = summary
    a = summary.attrs
    print(f'=== {run_dir.name} ===  {a["tasks"]} tasks, {a["rows"]:,} rows; '
          f'{time.perf_counter() - started:.0f} s')
    print(f'lowest held-out error at {a["best"]} components; one-SE rule: {a["one_se"]} components')
    if a['best_at_limit']:
        print(f'  the lowest error is at SELECTION_MAX_COMPONENTS = {SELECTION_MAX_COMPONENTS}; raise it to see where it turns.')
    print(summary.drop(columns='in_sample_mse').round(4).to_string())
    bs.plot_component_selection(summary, f'{run_dir.name}: held-out error by PLS components').show()
    gc.collect()

print('Recommended (one-SE rule), against what is configured now:')
for name, summary in selections.items():
    a, configured = summary.attrs, settings_for(name).pls_components
    print(f'  {name}: {a["one_se"]} (lowest error at {a["best"]}); configured {configured}, '
          f'held-out R2 {summary.loc[a["one_se"], "q2"]:.4f} at {a["one_se"]} '
          f'vs {summary.loc[configured, "q2"]:.4f} at {configured}')
print('PER_MODEL entries:', {name: dict(PLS_COMPONENTS=s.attrs['one_se']) for name, s in selections.items()})

## Fit helper

`fit_model` fits PLS and the surface for one model with its own settings (`settings_for`), prints the loss every iteration, the time taken and a short summary, then plots the rows and the fitted surface in PLS1-3.

`convex_height_min_eigen_ratio` is the smallest eigenvalue of the convex height's Hessian over the chart, relative to the median largest one: at or above about -0.05 the height is convex to the constraints' precision. `hull_chart_share` is the share of the chart box that the rows' convex hull covers.

In [5]:
def fit_model(run_dir):
    started = time.perf_counter()

    def progress(iteration, loss):
        print(f'  iteration {iteration:3d}: loss {loss:.6g}  ({time.perf_counter() - started:.1f} s)')

    settings = settings_for(run_dir.name)
    print(f'=== {run_dir.name} ===  PLS_COMPONENTS {settings.pls_components}, '
          f'REGULARIZATION {settings.surface.shape_bending:g}, '
          f'OUTER_WEIGHT {settings.surface.outer_weight:g}')
    result = bs.fit_shape(run_dir, settings, progress=progress)
    d = result.diagnostics
    print(f'{len(result.rows):,} rows; {d["iterations"]} iterations, converged: {d["converged"]}; '
          f'{time.perf_counter() - started:.1f} s including loading and PLS')
    print(f'final loss {d["objective"]:.6g} (start {d["objective_start"]:.6g})')
    print(d[['weighted_rms_distance', 'variance_share_on_surface', 'bending_energy',
             'rows_on_patch_boundary', 'residual_tangent_cosine_max',
             'convex_height_min_eigen_ratio', 'hull_chart_share', 'fit_weight_radius_ratio']].to_string())
    bs.plot_shape(result, PLOT_ROWS).show()
    return result

## Fit every model with a BCPC bundle

One model failing does not stop the others. Its traceback is printed, and once every model has run the cell raises.

In [6]:
fits, failures = {}, {}
for run_dir in run_dirs:
    try:
        fits[run_dir.name] = fit_model(run_dir)
    except Exception:
        failures[run_dir.name] = traceback.format_exc()
        print(failures[run_dir.name])
    gc.collect()

if failures:
    raise RuntimeError(f'Fit failures: {list(failures)}')

=== gemma-4-31B-it ===  PLS_COMPONENTS 16, REGULARIZATION 5e-05, OUTER_WEIGHT 1
  iteration   1: loss 441.692  (32.7 s)
  iteration   2: loss 434.059  (33.5 s)
  iteration   3: loss 431.701  (34.2 s)
  iteration   4: loss 430.962  (35.0 s)
  iteration   5: loss 430.677  (35.6 s)
  iteration   6: loss 430.553  (36.3 s)
  iteration   7: loss 430.493  (36.9 s)
  iteration   8: loss 430.453  (37.6 s)
  iteration   9: loss 430.433  (38.2 s)
  iteration  10: loss 430.417  (38.7 s)
  iteration  11: loss 430.41  (39.3 s)
  iteration  12: loss 430.401  (39.9 s)
  iteration  13: loss 430.398  (40.5 s)
  iteration  14: loss 430.392  (41.0 s)
  iteration  15: loss 430.391  (41.6 s)
  iteration  16: loss 430.388  (42.2 s)
  iteration  17: loss 430.387  (42.8 s)
  iteration  18: loss 430.387  (43.4 s)
15,912 rows; 19 iterations, converged: True; 45.0 s including loading and PLS
final loss 430.03 (start 480.232)
weighted_rms_distance                20.070463
variance_share_on_surface             0.67

=== Mistral-Small-3.1-24B-Instruct-2503 ===  PLS_COMPONENTS 16, REGULARIZATION 5e-05, OUTER_WEIGHT 1
  iteration   1: loss 5.99914  (26.5 s)
  iteration   2: loss 5.9191  (27.2 s)
  iteration   3: loss 5.87937  (28.0 s)
  iteration   4: loss 5.86097  (28.7 s)
  iteration   5: loss 5.85229  (29.4 s)
  iteration   6: loss 5.84764  (30.1 s)
  iteration   7: loss 5.84504  (30.8 s)
  iteration   8: loss 5.84355  (31.5 s)
  iteration   9: loss 5.8427  (32.1 s)
  iteration  10: loss 5.84217  (32.8 s)
  iteration  11: loss 5.84186  (33.4 s)
  iteration  12: loss 5.84167  (34.0 s)
  iteration  13: loss 5.84155  (34.6 s)
  iteration  14: loss 5.84142  (35.2 s)
  iteration  15: loss 5.84136  (35.8 s)
  iteration  16: loss 5.84131  (36.4 s)
  iteration  17: loss 5.84127  (37.0 s)
  iteration  18: loss 5.84124  (37.7 s)
  iteration  19: loss 5.84112  (38.3 s)
  iteration  20: loss 5.84111  (38.9 s)
  iteration  21: loss 5.84097  (39.4 s)
  iteration  22: loss 5.84094  (40.0 s)
  iteration  23: loss

=== Qwen3-32B ===  PLS_COMPONENTS 16, REGULARIZATION 5e-05, OUTER_WEIGHT 1
  iteration   1: loss 595.288  (28.5 s)
  iteration   2: loss 577.073  (29.5 s)
  iteration   3: loss 573.297  (30.3 s)
  iteration   4: loss 571.887  (31.1 s)
  iteration   5: loss 571.197  (31.8 s)
  iteration   6: loss 570.818  (32.6 s)
  iteration   7: loss 570.59  (33.3 s)
  iteration   8: loss 570.428  (34.1 s)
  iteration   9: loss 570.294  (34.8 s)
  iteration  10: loss 570.197  (35.4 s)
  iteration  11: loss 570.091  (36.1 s)
  iteration  12: loss 569.953  (36.7 s)
  iteration  13: loss 569.861  (37.4 s)
  iteration  14: loss 569.79  (38.0 s)
  iteration  15: loss 569.748  (38.6 s)
  iteration  16: loss 569.701  (39.3 s)
  iteration  17: loss 569.646  (40.0 s)
  iteration  18: loss 569.576  (40.7 s)
  iteration  19: loss 569.488  (41.3 s)
  iteration  20: loss 569.388  (42.0 s)
  iteration  21: loss 569.285  (42.7 s)
  iteration  22: loss 569.185  (43.4 s)
  iteration  23: loss 569.112  (44.0 s)
  itera

## Save the fitted surfaces

Writes each fitted model's surface to `artifacts/<model>/pls/shape/`:

- `model.npz`: the PLS transform (`pls_mean`, `pls_rotations`, ...) and the surface (`shape_*`: origin, frame, chart box, B-spline heights and the rows' convex hull)
- `model.json`: settings (including that model's `REGULARIZATION` and `OUTER_WEIGHT`), fit diagnostics, checksums of the files, the code and the BCPC bundle it was fitted on
- `rows.parquet`: every row with its PLS scores and closest point on the surface (`surface_s`, `surface_t`, `surface_distance`); no stakes classes
- `shape.html`: the plot above

Each saved surface is reloaded and must reproduce a sample of rows from their raw activations. `bs.load_shape(config)` reads it back. The older full surface in `pls/` itself (u map and (u, v) coordinates) is left untouched.

In [9]:
for name, result in fits.items():
    directory = bs.save_shape(result, notebook='notebooks/3_pls_arc_length_surface.ipynb', plot_rows=PLOT_ROWS)
    print(f'{name}: saved to {directory}')

gemma-4-31B-it: saved to C:\Users\91967\Desktop\AISC\stakes_manifold\artifacts\content\artifacts\gemma-4-31B-it\pls\shape
Mistral-Small-3.1-24B-Instruct-2503: saved to C:\Users\91967\Desktop\AISC\stakes_manifold\artifacts\content\artifacts\Mistral-Small-3.1-24B-Instruct-2503\pls\shape
Qwen3-32B: saved to C:\Users\91967\Desktop\AISC\stakes_manifold\artifacts\content\artifacts\Qwen3-32B\pls\shape
